# 03 — Explanation Faithfulness (D7–D9)

**This notebook is the paper.** Detection accuracy is table stakes — ≥5 YOLO-family works already exist on VinDr-CXR. What no verified prior work does is *quantify* whether the model looks where the radiologist looked.

The asset: **VinDr ships ground-truth boxes.** Most CXR-XAI work uses NIH ChestX-ray14, which has none and can therefore only show qualitative heatmaps.

> ⚠️ **The trap:** "we ran three YOLOs and made heatmaps" is a course project — literally what the vendored `tariqshaban` repo already is. The measured saliency evaluation is the whole difference.
>
> **If the schedule slips, cut a model. Never cut these metrics.**

**Gate D8:** if metrics can't be computed reliably across all models, drop to 2 (YOLOv8s vs YOLO26s — the NMS vs NMS-free contrast, Axis B) and keep the full analysis.

---
**Prerequisite:** Add Data → "Notebook Output Files" → "Your Work" → attach **both** notebook 01 (dataset) and notebook 02 (weights). Outputs mount at `/kaggle/input/notebooks/<username>/<slug>/...`; paths are discovered, not hardcoded.

**Accelerator:** `GPU T4 x2`, `device=0`. P100 is sm_60 and unusable — see notebook 02.

In [ ]:
!pip install -q ultralytics grad-cam

REPO = "/kaggle/working/repo"
!rm -rf {REPO} && git clone -q https://github.com/ShMazumder/vinbigdata-chest-x-ray.git {REPO}

import sys
sys.path.insert(0, REPO)

from pathlib import Path
import pandas as pd, numpy as np
from ultralytics import YOLO

from src import config as C
from src.evaluation import xai
from src.utils.run_logger import capture_gpu

!cd {REPO} && git rev-parse --short HEAD
capture_gpu(strict=True)

In [ ]:
DATA_YAML = C.find_data_yaml()
# Fallback if the search misses -- same validation runs:
# DATA_YAML = C.find_data_yaml(
#     "/kaggle/input/notebooks/<username>/01-data-prep/vindr_yolo/data.yaml")

DATA_ROOT = DATA_YAML.parent
TEST_IMG = DATA_ROOT / "images" / "test"
TEST_LBL = DATA_ROOT / "labels" / "test"

WEIGHTS = C.find_weights()
models = {k: YOLO(v) for k, v in WEIGHTS.items()}
print(f"\n{len(models)} model(s) loaded: {list(models)}")

## 1. Target layer check — DO THIS FIRST

> ⚠️ **The fragile part.** Target-layer selection differs across v8/v11/YOLO26. `pick_target_layer()` takes the last `Conv2d`, which may land *inside* the detection head rather than before it.
>
> **Verify the printed layer is pre-head.** A CAM from the wrong layer produces plausible-looking garbage — it won't error, it'll just be wrong, and every number downstream inherits it.

In [ ]:
for k, m in models.items():
    print(f"\n--- {k} ---")
    xai.pick_target_layer(m)

## 2. Qualitative check (D7)

Look at CAMs overlaid on GT boxes **before** running any batch job. If they're obviously wrong, the batch run just spends hours producing wrong numbers.

Same lesson as notebook 01's fusion check — the cheap visual pass catches what the automated gate doesn't. Runs 1 and 2 of fusion both passed `verify()` while producing boxes stacked 8 deep.

In [ ]:
import cv2, matplotlib.pyplot as plt, matplotlib.patches as patches
from src.evaluation.metrics import ground_truth_from_yolo

ref = "yolo26s" if "yolo26s" in models else list(models)[0]
gts = ground_truth_from_yolo(TEST_LBL, TEST_IMG)
sample = sorted(TEST_IMG.glob("*.jpg"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, p in zip(axes.ravel(), sample):
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    cam = xai.compute_cam(models[ref], img, method="eigencam", imgsz=C.IMGSZ)
    ax.imshow(cv2.resize(img, (C.IMGSZ, C.IMGSZ)), cmap="gray")
    ax.imshow(cam, cmap="jet", alpha=0.45)
    for _, r in gts[gts.image_id == p.stem].iterrows():
        sx, sy = C.IMGSZ / w, C.IMGSZ / h
        ax.add_patch(patches.Rectangle(
            (r.x1 * sx, r.y1 * sy), (r.x2 - r.x1) * sx, (r.y2 - r.y1) * sy,
            fill=False, edgecolor="lime", linewidth=2))
    ax.set_title(p.stem, fontsize=9); ax.axis("off")
plt.suptitle(f"{ref} — EigenCAM vs radiologist boxes", y=1.01)
plt.tight_layout()

## 3. Batch metrics (D8)

| Metric | What it measures |
|---|---|
| `pointing_game` | CAM argmax inside GT box? Coarse but standard. |
| `ebpg` | Fraction of CAM **energy** inside GT box. **Lead with this** — uses the whole map, so it separates "peaked correctly by luck" from "consistently attends". |
| `cam_box_iou` | IoU of 90th-percentile-thresholded CAM vs GT |

In [ ]:
all_xai = []
for key, model in models.items():
    for method in C.XAI_METHODS:
        print(f"\n=== {key} / {method} ===")
        df = xai.evaluate_xai(model, TEST_IMG, TEST_LBL, method=method,
                              imgsz=C.IMGSZ, n_images=C.XAI_N_IMAGES)
        df["model"] = key
        all_xai.append(df)

xai_df = pd.concat(all_xai, ignore_index=True)
C.RUNS_DIR.mkdir(parents=True, exist_ok=True)
xai_df.to_csv(C.RUNS_DIR / "xai_metrics.csv", index=False)
print(f"\n{len(xai_df)} rows")

## 4. Axis A — small vs large targets

**Hypothesis:** explanation faithfulness degrades on small targets *faster than mAP does*.

If true, that's clinically meaningful — **the model looks right while explaining wrong**, precisely on the classes (Nodule/Mass 39.69% small, Calcification 17.67%) where a radiologist most needs the explanation.

In [ ]:
for key in models:
    sub = xai_df[(xai_df.model == key) & (xai_df.method == "eigencam")]
    print(f"\n=== {key} ===")
    print(xai.summarize_xai(sub)["by_target_size"])

In [ ]:
sub = xai_df[xai_df.method == "eigencam"].copy()
sub["area_bin"] = pd.qcut(sub.box_area_frac, 5, labels=["XS","S","M","L","XL"])
pivot = sub.pivot_table(index="area_bin", columns="model", values="ebpg", observed=False)
pivot.plot(marker="o", figsize=(8,5), title="EBPG vs object size")
plt.ylabel("Energy-based pointing game"); plt.grid(alpha=.3)
pivot.round(4)

## 5. Axis B — NMS-free vs NMS

**This ties the two halves of the paper together.** Does YOLO26's native NMS-free head produce more faithful explanations than v8/v11?

Without this you have two unrelated experiments stapled together — which is why, if models have to be cut, YOLOv8s and YOLO26s are the two to keep.

In [ ]:
axis_b = xai_df.groupby(["model", "method"])[
    ["pointing_game", "ebpg", "cam_box_iou"]
].mean().round(4)
axis_b["nms_free"] = [i[0] == "yolo26s" for i in axis_b.index]
axis_b

## 6. Axis C — accuracy/faithfulness decoupling

Does the highest-mAP model give the best explanations?

**A "no" is the most publishable outcome available.** It says accuracy and explanation quality are separate axes — which is exactly the argument for measuring the second rather than assuming it follows the first.

In [ ]:
import json

cands = list(Path("/kaggle/input").glob("**/eval_summary.json"))
assert cands, "attach notebook 02's output"
ev = json.loads(cands[0].read_text())
print(f"loaded {cands[0]}")

acc = pd.DataFrame({k: {"mAP@0.4": v["map40"], "mAP@0.5": v["map50"]}
                    for k, v in ev.items()}).T
fai = xai_df[xai_df.method == "eigencam"].groupby("model")[["ebpg", "pointing_game"]].mean()

combined = acc.join(fai).round(4)
combined["rank_mAP"] = combined["mAP@0.4"].rank(ascending=False).astype(int)
combined["rank_EBPG"] = combined["ebpg"].rank(ascending=False).astype(int)
combined["DECOUPLED"] = combined.rank_mAP != combined.rank_EBPG
combined

## 7. Causal faithfulness — optional, D9 only if ahead

Deletion/insertion AUC needs ~40× the forward passes. **Subsample only.**

A saliency map that looks convincing but has flat deletion AUC isn't explaining the model — precisely the failure MedFocus / MedGround-Bench reported for standard attribution on VinDr.

In [ ]:
RUN_CAUSAL = False   # flip only if D9 is on schedule

if RUN_CAUSAL:
    causal = xai.evaluate_xai(models[ref], TEST_IMG, TEST_LBL,
                              method="eigencam", imgsz=C.IMGSZ,
                              n_images=50, with_causal=True, causal_subsample=50)
    causal.to_csv(C.RUNS_DIR / "xai_causal.csv", index=False)
    print(causal[["deletion_auc", "insertion_auc"]].mean())

## 8. Export for the paper (D10)

Feed these into the `latex-results-table` skill for booktabs formatting.

In [ ]:
combined.to_csv(C.RUNS_DIR / "table_main.csv")
axis_b.to_csv(C.RUNS_DIR / "table_axis_b.csv")
pivot.to_csv(C.RUNS_DIR / "table_axis_a.csv")

print("tables written. Write-up reminders:")
print("  - report BOTH mAP@0.4 and mAP@0.5")
print("  - do NOT claim SOTA (bar: RT-DETR 45.3, YOLOv11-MFF 41.5, YOLO-CXR 0.338)")
print("  - limitations: 512px, s-scale, single fold, positive-only, 40 epochs,")
print("    95.5% of labels from 3 radiologists, WBF@0.25 fusion, T4")
print("  - Atelectasis n=22 / Pneumothorax n=10: report per-class AP with n")